[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Indexes


## What you will be able to do

Create indexes, and read `explain` well enough to know whether one is being used. Say why a compound
index on `(kind, price)` answers a query about `kind` and does nothing at all for a query about
`price`. Recognize the two scans that look fast and are not: a collection scan, and an index scan
that reads every key in the index. Build a unique index over data that already has duplicates, and
read the failure. And know why a TTL index does not delete anything at the moment it expires.


## The idea

### The problem

Every query in this guide so far has run against five hundred documents, where the difference
between a good plan and a terrible one is invisible. This notebook seeds two hundred thousand,
because a collection scan and an index scan take about the same time over five hundred documents and
differ by a factor of a hundred over two hundred thousand.

### What an index is

A second, sorted copy of one or more fields, with a pointer back to the document. Finding a value in
it is a walk down a tree rather than a look at every document. It costs disk, it costs a little on
every write, and it is the difference between a query that works and one that works for now.

### Why the order of a compound index matters

An index on `(kind, price)` is sorted by `kind` first. Every entry for `laptop` is together, and
within them they are sorted by price. So it answers "laptops", and "laptops under 100", and
"anything, sorted by kind". It cannot answer "anything under 100", because the prices are scattered
through it in five separate runs.

### Where this shows up

The first time a collection gets big. Everything works in development, nothing changes in the code,
and one query starts taking four seconds because nobody ever ran `explain` on it.

### What this notebook covers

`create_index`, `list_indexes`, `drop_index`. Reading `explain`: `COLLSCAN`, `IXSCAN`, and the
numbers that matter. Compound order and prefixes. Regexes, anchored and not. Text indexes and
`$text`. Unique indexes over dirty data, and the two conflicts. TTL, and its delay.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import pymongo

client = pymongo.MongoClient("mongodb://127.0.0.1:27017/shop", tz_aware=True)
shop = client.get_default_database()

for index in list(shop.products.list_indexes()):                # start from no index but _id
    if index["name"] != "_id_":
        shop.products.drop_index(index["name"])


def plan(cursor):
    explained = cursor.explain()
    winner = explained["queryPlanner"]["winningPlan"]
    stats = explained["executionStats"]
    return (winner.get("inputStage", winner).get("stage"),
            stats["totalDocsExamined"], stats["nReturned"])


query = {"kind": "laptop", "price": {"$lt": 100}}
print("before an index:", plan(shop.products.find(query)))

shop.products.create_index([("kind", 1), ("price", 1)], name="kind_price")
print("after an index: ", plan(shop.products.find(query)))
print("the same answer, read from 200000 documents or from 1889")
client.close()
```

```
before an index: ('COLLSCAN', 200000, 1889)
after an index:  ('IXSCAN', 1889, 1889)
the same answer, read from 200000 documents or from 1889
```

The query did not change and the answer did not change. What changed is that the server stopped
reading every document in the collection to find the eighteen hundred it wanted.


## Setup

Eight imports, MongoDB, the boot cell, and three helpers.

- `pymongo` is the driver, `datetime` is for the TTL section, and `time` waits for its sweep
- `subprocess` and `os` install and start the server, `sys` names this Python
- `random` seeds the data the same way every run, with `version` and `PackageNotFoundError`

**This Setup seeds two hundred thousand documents rather than five hundred**, which takes a few
seconds and is the whole reason the numbers below mean anything. `plan` runs `explain` and pulls out
the four values worth reading. `only_id` drops every index but `_id`'s, so each section starts from
a known state. `failed` prints the stable part of a failure's message.


In [1]:
import datetime as dt
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pymongo

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

def failed(error):
    """The part of a failure that is the same on every run. An index build failure names two
    fresh uuids before the real message, and the real message is after 'caused by'."""
    details = getattr(error, "details", None) or {}
    message = details.get("errmsg", str(error).split(", full error")[0])
    return f"{type(error).__name__}: {message.split(' :: caused by :: ')[-1]}"


def plan(cursor):
    """How the server answered: the stage it used and how much it had to look at."""
    explained = cursor.explain()
    winner = explained["queryPlanner"]["winningPlan"]
    stats = explained["executionStats"]
    return {"stage": winner.get("inputStage", winner).get("stage"),
            "index keys": stats["totalKeysExamined"],
            "documents": stats["totalDocsExamined"],
            "returned": stats["nReturned"]}


def only_id(collection):
    """Drop every index but the one on _id, so a section can start from nothing."""
    for index in list(collection.list_indexes()):
        if index["name"] != "_id_":
            collection.drop_index(index["name"])
    return [index["name"] for index in collection.list_indexes()]


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(200_000), "products   <- this notebook needs a big collection")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  200000 products   <- this notebook needs a big collection
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 200000


## Worked examples

### What a collection scan looks like


In [2]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
print("indexes now:", only_id(shop.products))

print("kind = laptop:", plan(shop.products.find({"kind": "laptop"})))


indexes now: ['_id_']
kind = laptop: {'stage': 'COLLSCAN', 'index keys': 0, 'documents': 200000, 'returned': 40000}


`COLLSCAN`, two hundred thousand documents examined, forty thousand returned. The server read every
document in the collection and threw away four fifths of them.

`documents` against `returned` is the ratio to look at. One to one is ideal; five to one is this
query; two hundred to one is a query that will fall over.

### And what an index does


In [3]:
shop.products.create_index([("kind", 1), ("price", 1)], name="kind_price")

print("kind = laptop:            ", plan(shop.products.find({"kind": "laptop"})))
print("kind = laptop, price < 100:", plan(shop.products.find({"kind": "laptop",
                                                              "price": {"$lt": 100}})))


kind = laptop:             {'stage': 'IXSCAN', 'index keys': 40000, 'documents': 40000, 'returned': 40000}
kind = laptop, price < 100: {'stage': 'IXSCAN', 'index keys': 1889, 'documents': 1889, 'returned': 1889}


The second one is what a good index looks like: `index keys`, `documents` and `returned` are all the
same number, so every entry the server touched turned into a row of the answer.

The first is honest too. There really are forty thousand laptops, so reading forty thousand is not
waste. An index cannot make a large answer small.

### Compound order, and the query it cannot help


In [4]:
print("on kind:          ", plan(shop.products.find({"kind": "mouse"})))
print("on kind and price:", plan(shop.products.find({"kind": "mouse", "price": {"$lt": 100}})))
print("on price alone:   ", plan(shop.products.find({"price": {"$lt": 100}})))


on kind:           {'stage': 'IXSCAN', 'index keys': 40000, 'documents': 40000, 'returned': 40000}
on kind and price: {'stage': 'IXSCAN', 'index keys': 1901, 'documents': 1901, 'returned': 1901}
on price alone:    {'stage': 'COLLSCAN', 'index keys': 0, 'documents': 200000, 'returned': 9509}


The third is a `COLLSCAN`, with the index sitting right there. An index on `(kind, price)` is sorted
by `kind`, so the documents under a hundred are in five separate stretches of it and there is no way
to walk to them.

The rule is the **prefix** rule: an index on `(a, b, c)` helps a query on `a`, on `a` and `b`, and
on `a`, `b` and `c`. It does not help a query on `b` alone.


In [5]:
shop.products.create_index("price", name="price_1")
print("on price alone, now:", plan(shop.products.find({"price": {"$lt": 100}})))
print()
print("indexes on this collection:", [index["name"] for index in shop.products.list_indexes()])


on price alone, now: {'stage': 'IXSCAN', 'index keys': 9509, 'documents': 9509, 'returned': 9509}

indexes on this collection: ['_id_', 'kind_price', 'price_1']


Which is why the order in `create_index([("kind", 1), ("price", 1)])` is a decision rather than a
formality: put the field you always filter on first, and the one you filter on sometimes, or sort
by, second.

### Sorting, and the index that provides it


In [6]:
by_price = shop.products.find({"kind": "laptop"}).sort("price").limit(5)
print("sorted by price, inside a kind:", plan(by_price))

by_stock = shop.products.find({"kind": "laptop"}).sort("stock").limit(5)
print("sorted by a field not in the index:", plan(by_stock))


sorted by price, inside a kind: {'stage': 'FETCH', 'index keys': 5, 'documents': 5, 'returned': 5}
sorted by a field not in the index: {'stage': 'FETCH', 'index keys': 40000, 'documents': 40000, 'returned': 5}


Five keys read against forty thousand, for the same five documents. The first query needs no sorting
at all, because the index already holds the laptops in price order, so `limit(5)` walks five
entries and stops. The second has to collect all forty thousand laptops and sort them before it can
know which five are first, and sorting that many in memory has a fixed budget it can exceed.

That is the second reason compound order matters: an index can supply the sort as well as the
filter, and only in the order it was built.

### Regexes


In [7]:
shop.products.create_index("name", name="name_1")

print("anchored at the start:", plan(shop.products.find({"name": {"$regex": "^Aster laptop 12"}})))
print("not anchored:         ", plan(shop.products.find({"name": {"$regex": "laptop 12"}})))


anchored at the start: {'stage': 'IXSCAN', 'index keys': 557, 'documents': 556, 'returned': 556}
not anchored:          {'stage': 'IXSCAN', 'index keys': 200000, 'documents': 2222, 'returned': 2222}


Both say `IXSCAN`, and only one of them is fast. The unanchored one read every key in the index,
two hundred thousand of them, because a pattern that can match anywhere in the string gives the
index nothing to start from.

This is the most misleading thing `explain` will tell you. `IXSCAN` is not a pass mark; the numbers
are. An index scan that reads the whole index has replaced a collection scan with something no
better and sometimes worse.

For searching inside text, that is what a text index is for:


In [8]:
shop.notes.drop()
shop.notes.insert_many([{"body": "a fast red laptop"},
                        {"body": "a slow blue monitor"},
                        {"body": "a laptop stand, in red"}])
shop.notes.create_index([("body", "text")])

for note in shop.notes.find({"$text": {"$search": "laptop"}}, {"_id": 0}):
    print(" ", note["body"])
print()
print("a text index holds words, so it can find them in the middle of a string")


  a laptop stand, in red
  a fast red laptop

a text index holds words, so it can find them in the middle of a string


A collection may have **one** text index, covering as many fields as you like. Asking for a second
one is one of the conflicts below.

### Unique indexes


In [9]:
shop.people.drop()
shop.people.insert_many([{"email": "a@example.com"}, {"email": "b@example.com"}])
shop.people.create_index("email", unique=True)

print("built:", [index["name"] for index in shop.people.list_indexes()])
try:
    shop.people.insert_one({"email": "a@example.com"})
except pymongo.errors.DuplicateKeyError as error:
    print("and it now refuses a repeat:", failed(error))


built: ['_id_', 'email_1']
and it now refuses a repeat: DuplicateKeyError: E11000 duplicate key error collection: shop.people index: email_1 dup key: { email: "a@example.com" }


A unique index is a constraint that happens to be an index. It is the only way to say "no two
documents may have the same value here", and it is enforced from the moment it is built.

### TTL


In [10]:
shop.sessions.drop()
shop.sessions.create_index("at", expireAfterSeconds=1)
shop.sessions.insert_one({"at": dt.datetime.now(dt.timezone.utc) - dt.timedelta(hours=1)})

print("the document is an hour past its expiry:", shop.sessions.count_documents({}))
print("index:", [index["name"] for index in shop.sessions.list_indexes()
                 if index["name"] != "_id_"])
print()
print("nothing has been deleted, and nothing is wrong")


the document is an hour past its expiry: 1
index: ['at_1']

nothing has been deleted, and nothing is wrong


A TTL index does not delete anything when a document expires. A background task sweeps the
collection about once a minute and deletes what it finds, so an expired document stays readable for
up to a minute after its time, and longer if the server is busy.

That is documented and it is still read as a bug. If your code must never see an expired document,
filter on the date as well as relying on the index.

### When to reach for which

| What you want | How to write it |
|---|---|
| an index on one field | `create_index("price")` |
| an index on several | `create_index([("kind", 1), ("price", 1)])` |
| to know whether it is used | `explain`, and read the numbers, not the stage name |
| a filter plus a sort | put the filter field first and the sort field second |
| no duplicates | `create_index("email", unique=True)` |
| to search inside text | a text index and `$text`, one per collection |
| a prefix match | an anchored regex, `^like this` |
| documents to expire | `create_index("at", expireAfterSeconds=...)`, and wait |
| to remove one | `drop_index("name")` |

The default is no index beyond `_id` until a query needs one. Reach for `explain` before you reach
for `create_index`, because the index you guess at is often not the one the query wants.

### Indexing a collection for the queries it actually serves, finished


In [11]:
def report_on(shop, queries):
    """Run explain over a set of queries and say which ones are reading too much."""
    rows = []
    for label, cursor_maker in queries.items():
        measured = plan(cursor_maker())
        waste = (measured["documents"] / measured["returned"]) if measured["returned"] else 0
        rows.append((label, measured["stage"], measured["documents"], measured["returned"],
                     round(waste, 1)))
    return rows


queries = {
    "one kind":        lambda: shop.products.find({"kind": "cable"}),
    "kind and price":  lambda: shop.products.find({"kind": "cable", "price": {"$lt": 50}}),
    "price alone":     lambda: shop.products.find({"price": {"$lt": 50}}),
    "one maker":       lambda: shop.products.find({"maker": "Aster"}),
    "sku, exact":      lambda: shop.products.find({"sku": "CAB-000004"}),
}

only_id(shop.products)
print("with no indexes at all:")
for row in report_on(shop, queries):
    print(f"  {row[0]:16} {row[1]:9} read {row[2]:6} returned {row[3]:6} waste {row[4]}")

shop.products.create_index([("kind", 1), ("price", 1)], name="kind_price")
shop.products.create_index("maker", name="maker_1")
shop.products.create_index("sku", name="sku_1", unique=True)

print()
print("with three indexes:")
for row in report_on(shop, queries):
    print(f"  {row[0]:16} {row[1]:9} read {row[2]:6} returned {row[3]:6} waste {row[4]}")


with no indexes at all:
  one kind         COLLSCAN  read 200000 returned  40000 waste 5.0
  kind and price   COLLSCAN  read 200000 returned    980 waste 204.1
  price alone      COLLSCAN  read 200000 returned   4480 waste 44.6
  one maker        COLLSCAN  read 200000 returned  50000 waste 4.0
  sku, exact       COLLSCAN  read 200000 returned      1 waste 200000.0

with three indexes:
  one kind         IXSCAN    read  40000 returned  40000 waste 1.0
  kind and price   IXSCAN    read    980 returned    980 waste 1.0
  price alone      COLLSCAN  read 200000 returned   4480 waste 44.6
  one maker        IXSCAN    read  50000 returned  50000 waste 1.0
  sku, exact       EXPRESS_IXSCAN read      1 returned      1 waste 1.0


Read the `waste` column. Anything near one is a query the index answers exactly. `price alone` is
still a collection scan, deliberately: it returns tens of thousands of documents, so an index would
save nothing worth the write cost, and the honest answer to that query is to ask a narrower one.

The `sku` index is unique as well as fast, which is two jobs from one index and the usual reason to
make a natural key unique.

### Where each part came from

| In `report_on` | What it relies on | The section that showed it |
|---|---|---|
| `plan(cursor)` | `explain` with execution statistics | What a collection scan looks like |
| `documents / returned` | the ratio that says how much was wasted | What a collection scan looks like |
| the compound index | filter field first, range field second | Compound order |
| `unique=True` on `sku` | a constraint that is also an index | Unique indexes |
| `only_id` before measuring | a known starting state | Setup |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/08-indexes-solutions.ipynb).

**1.** Drop every index but `_id`'s and explain a query on `maker`.


In [12]:
# your code here


**2.** Add an index on `maker` and explain the same query again.


In [13]:
# your code here


**3.** Build a compound index on `(maker, stock)` and show it does not help a query on `stock`.


In [14]:
# your code here


**4.** Compare an anchored regex with an unanchored one on `name`.


In [15]:
# your code here


**5.** Make `sku` unique, then try to insert a duplicate.


In [16]:
# your code here


**6.** Create a TTL index and show the document is still there.


In [17]:
# your code here


## Common errors

### pymongo.errors.DuplicateKeyError: E11000 duplicate key error


In [18]:
shop.dirty.drop()
shop.dirty.insert_many([{"email": "a@example.com"}, {"email": "a@example.com"},
                        {"email": "b@example.com"}])

try:
    shop.dirty.create_index("email", unique=True)
except pymongo.errors.DuplicateKeyError as error:
    print(failed(error))


DuplicateKeyError: E11000 duplicate key error collection: shop.dirty index: email_1 dup key: { email: "a@example.com" }


The index was not built, and the collection is exactly as it was. A unique index cannot be created
over data that already violates it, which is the right behavior and is always discovered at the
worst moment, because the data got dirty before anybody thought to add the constraint.

The message names one offending value. Finding all of them is an aggregation, which is what
**The Aggregation Pipeline** is about, but the shape is worth having now:


In [19]:
duplicates = list(shop.dirty.aggregate([
    {"$group": {"_id": "$email", "n": {"$sum": 1}}},
    {"$match": {"n": {"$gt": 1}}},
]))
print("every duplicated value:", duplicates)

shop.dirty.delete_one({"email": "a@example.com"})                   # clean, then build
shop.dirty.create_index("email", unique=True)
print("built now:", [index["name"] for index in shop.dirty.list_indexes()])


every duplicated value: [{'_id': 'a@example.com', 'n': 2}]
built now: ['_id_', 'email_1']


### pymongo.errors.OperationFailure: An existing index has the same name as the requested index


In [20]:
shop.conflict.drop()
shop.conflict.insert_one({"k": 1})
shop.conflict.create_index("k")                                     # named k_1 automatically

try:
    shop.conflict.create_index("k", unique=True)                    # also wants to be called k_1
except pymongo.errors.OperationFailure as error:
    print(failed(error).split(". When index")[0])
    print("  codeName:", error.details["codeName"])


OperationFailure: An existing index has the same name as the requested index
  codeName: IndexKeySpecsConflict


Two different indexes cannot share a name, and PyMongo generates the name from the keys, so a second
index on the same keys with different options collides before the options are even considered.

Give it a name and you get the other conflict, which is the one that says what is really wrong:


In [21]:
try:
    shop.conflict.create_index("k", unique=True, name="k_unique")
except pymongo.errors.OperationFailure as error:
    print(failed(error))
    print("  codeName:", error.details["codeName"])

print()
print("creating the identical index again is a no-op:", shop.conflict.create_index("k"))



creating the identical index again is a no-op: k_1


`IndexOptionsConflict` means an index on those keys already exists and yours differs. To change an
index's options you drop it and build it again; there is no altering one in place.

Note the last line: asking for an index that already exists, with the same options, does nothing and
raises nothing. That is what makes `create_index` safe to call at startup.

### No error: IXSCAN that reads the whole index


In [22]:
only_id(shop.products)
shop.products.create_index("name", name="name_1")

print("anchored:  ", plan(shop.products.find({"name": {"$regex": "^Aster laptop 12"}})))
print("unanchored:", plan(shop.products.find({"name": {"$regex": "laptop 12"}})))


anchored:   {'stage': 'IXSCAN', 'index keys': 557, 'documents': 556, 'returned': 556}
unanchored: {'stage': 'IXSCAN', 'index keys': 200000, 'documents': 2222, 'returned': 2222}


Both are `IXSCAN`. One examined a few hundred keys and the other examined two hundred thousand, and
a dashboard that counts collection scans would call them both fine.

The number to watch is `index keys` against `returned`. When the first is the size of the
collection, the index is being read end to end and is doing the job of a collection scan with extra
steps.

### No error: the TTL index that has not deleted anything


In [23]:
shop.sessions.drop()
shop.sessions.create_index("at", expireAfterSeconds=1)
shop.sessions.insert_many([
    {"_id": number, "at": dt.datetime.now(dt.timezone.utc) - dt.timedelta(days=1)}
    for number in range(3)
])

print("three documents, each a day past expiry:", shop.sessions.count_documents({}))
time.sleep(5)
print("five seconds later:", shop.sessions.count_documents({}), "<- the sweep runs about once a minute")


three documents, each a day past expiry: 3
five seconds later: 3 <- the sweep runs about once a minute


Nothing is broken. The TTL monitor wakes roughly every sixty seconds, and between wakes an expired
document is an ordinary document: it is returned by queries, it is counted, and it is visible to
your code.

If that is not acceptable, say so in the query rather than waiting:


In [24]:
cutoff = dt.datetime.now(dt.timezone.utc) - dt.timedelta(seconds=1)
print("what the TTL index will eventually remove:", shop.sessions.count_documents({}))
print("what a careful query sees:",
      shop.sessions.count_documents({"at": {"$gt": cutoff}}))


what the TTL index will eventually remove: 3
what a careful query sees: 0


In [25]:
for name in ("notes", "people", "sessions", "dirty", "conflict"):
    shop[name].drop()
only_id(shop.products)
client.close()
print("tidied up and closed")


tidied up and closed


## Recap

- An index is a sorted copy of some fields. It costs space and write time and it is the difference
  between a query that scales and one that does not.
- `explain` tells you the stage and the numbers. Read the numbers: `documents` against `returned`
  says how much was wasted, and a ratio near one is what you want.
- A compound index on `(a, b)` answers queries on `a`, and on `a` and `b`. It does nothing for a
  query on `b` alone, which is the **prefix** rule.
- An index can supply a sort as well as a filter, in the order it was built.
- An anchored regex uses an index properly. An unanchored one still says `IXSCAN` and reads every
  key, which is why the stage name alone proves nothing.
- A collection may have one text index, over any number of fields, and `$text` searches it.
- A unique index cannot be built over data that already has duplicates; the failure names one of
  them and the index is not created.
- Two indexes cannot share a name, and one already on those keys with different options is
  `IndexOptionsConflict`. Creating an identical index again does nothing.
- A TTL index deletes on a sweep about once a minute, so expired documents remain visible until it
  runs.


## What is next

**The Aggregation Pipeline** is the other way to ask: `$match`, `$group` and `$unwind`, the stage
order that decides whether an index can be used at all, and the `$unwind` that quietly drops every
document whose array is empty.


---

&#8592; **Previous:** [Bulk Writes and Transactions](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/07-bulk-writes-and-transactions.ipynb)  &nbsp;·&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
